In [1]:
import os
import random
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
import logging
from datasets import Dataset


from sklearn.metrics import accuracy_score
from transformers import TrainingArguments
from transformers import Trainer, TrainerCallback
from transformers.trainer_pt_utils import _get_learning_rate
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer, GenerationConfig, AutoConfig

# os.environ["TOKENIZERS_PARALLELISM"] = "true"
#os.environ["CUDA_VISIBLE_DEVICES"]= "0"


# ## Seed 고정

In [2]:
def seed_everything(seed:int = 1004):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # current gpu seed
    torch.cuda.manual_seed_all(seed) # All gpu seed
    torch.backends.cudnn.deterministic = True  
    torch.backends.cudnn.benchmark = False  # True로 하면 gpu에 적합한 알고리즘을 선택함.

seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


# ## Config

cuda


In [3]:
MODEL_NAME = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"

In [4]:
config = AutoConfig.from_pretrained(MODEL_NAME,
                                   trust_remote_code=True)  

print(config)

ExaoneConfig {
  "activation_function": "silu",
  "architectures": [
    "ExaoneForCausalLM"
  ],
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "configuration_exaone.ExaoneConfig",
    "AutoModelForCausalLM": "modeling_exaone.ExaoneForCausalLM",
    "AutoModelForSequenceClassification": "modeling_exaone.ExaoneForSequenceClassification"
  },
  "bos_token_id": 1,
  "dtype": "float32",
  "embed_dropout": 0.0,
  "eos_token_id": 361,
  "head_dim": 128,
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "layer_norm_epsilon": 1e-05,
  "max_position_embeddings": 32768,
  "model_type": "exaone",
  "num_attention_heads": 32,
  "num_key_value_heads": 8,
  "num_layers": 32,
  "pad_token_id": 0,
  "rope_scaling": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 1000000.0,
  "tie_word_embeddings": false,
  "transformers_version": "

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,)  # 토크나이저는 학습되어 있는 것으로
tokenizer.model_max_length=8192

In [6]:
print(tokenizer)

GPT2TokenizerFast(name_or_path='LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct', vocab_size=102400, model_max_length=8192, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[BOS]', 'eos_token': '[|endofturn|]', 'unk_token': '[UNK]', 'pad_token': '[PAD]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[BOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[EOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("                               ", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	5: AddedToken("                              ", rstrip=False, lstrip=False, single_word=False, normalized=F

In [7]:
tokenizer.eos_token_id, tokenizer.pad_token_id

(361, 0)

In [8]:
print(tokenizer("Hello I'm World!")), print(tokenizer.decode(tokenizer.encode("Hello I'm World!")))

{'input_ids': [33381, 768, 368, 438, 5542, 362], 'attention_mask': [1, 1, 1, 1, 1, 1]}
Hello I'm World!


(None, None)

In [9]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    attn_implementation = "flash_attention_2"
    #device_map={"": 0}
)

model.config.use_cache = False

print(next(model.parameters()).dtype)
print(model)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

torch.bfloat16
ExaoneForCausalLM(
  (transformer): ExaoneModel(
    (wte): Embedding(102400, 4096, padding_idx=0)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-31): 32 x ExaoneBlock(
        (ln_1): ExaoneRMSNorm()
        (attn): ExaoneAttention(
          (attention): ExaoneFlashAttention(
            (rotary): ExaoneRotaryEmbedding()
            (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
            (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
            (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
            (out_proj): Linear(in_features=4096, out_features=4096, bias=False)
          )
        )
        (ln_2): ExaoneRMSNorm()
        (mlp): ExaoneGatedMLP(
          (c_fc_0): Linear(in_features=4096, out_features=14336, bias=False)
          (c_fc_1): Linear(in_features=4096, out_features=14336, bias=False)
          (c_proj): Linear(in_features=14336, out_features=4096, bias=False)
     

In [10]:
print(model.num_parameters())


# ## 데이터 준비

7818448896


### Data

In [11]:
from datasets import load_dataset, load_from_disk

# ko_lima = load_dataset('taeshahn/ko-lima', 'plain') # or load_dataset('taeshahn/ko-lima')
# ko_lima_vicuna = load_dataset('taeshahn/ko-lima', 'vicuna')

ko_lima_conv = load_from_disk('/home/work/Korean_Spoken/Spoken_LIMA_onlysingleturn')
ko_lima_multi_conv = load_from_disk('/home/work/Korean_Spoken/Spoken_LIMA_onlymultiturn/')

In [12]:
ko_lima_conv, ko_lima_multi_conv

(Dataset({
     features: ['text', 'label'],
     num_rows: 1000
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 30
 }))

In [13]:
ko_lima_conv['text'][:5]

[['뇌세포가 움직일 수 있나요? 여기서 이동이란 장거리 이동(가급적 뇌 내에서만)을 의미합니다.',
  '이 질문은 꽤 넓은 범위를 다루고 있어서, 뇌가 뉴런만 있는 게 아니라 신경교세포(지지세포)와 유사분열 전의 신경 줄기세포까지 포함된다는 점을 같이 봐야 해요. 그리고 동료 과학자들이 지적했듯이, 발달 중인 배아의 뇌는 성인 뇌와 많이 다르기 때문에 발달 단계도 되게 중요합니다.\n\n하지만 여러 논문을 살펴보면 이 질문에 대한 답은 의외로 단순해요: 네, 뇌세포는 이동합니다. 성체 뇌에서는 아교세포가 이동하고(Klämbt, 2009), 그중에서도 희돌기아교세포같은 경우엔 절연 수초를 만들기 위해 비교적 먼 거리의 표적 축삭돌기를 찾아 이동하기도 합니다(Tsai and Miller, 2002). 신경 줄기세포도 손상에 반응해 장거리로 이동하고(Imitola et al., 2004), 해마나 뇌실하 영역 같은 특정 줄기세포 자리에서 다른 영역으로 이동하는 게 보고돼 있어요(Clarke, 2003). 유사분열 후지만 아직 분화되지 않은 뉴런들이 어류의 성체 뇌에서 이동하는 것(Scott et al., 2012)과 포유류·비인간 영장류에서도 이동하는 것이 관찰된 것도 같은 맥락입니다(Sawada et al., 2011).\n\n그리고 당연히 신경교세포, 줄기세포, 뉴런 모두 배아 발달 과정에서도 이동해요. 특히 말초 쪽으로 기능을 수행해야 하는 유사분열 후 뉴런들은 신경능선에서 목표 위치까지 비교적 먼 거리를 이동해야 합니다(신경과학, 2판, 뉴런의 이동).'],
 ['컴퓨터 시스템 강의에서 MIPS 프로세서에 대해 소개했습니다. 이 프로세서는 한 학기 동안 (재)개발되었으며 실제로 이해하기 매우 쉬웠습니다. 기본 명령어가 규칙적으로 인코딩되어 있고 전선을 단순하게 유지하기 위해 명령어가 몇 개만 있는 RISC 설계를 사용합니다.\nCISC는 다른 철학을 따른다고 언급했습니다. x86 명령어 세트를 잠깐 살펴보고 충격을 받았습니다. 이렇게 복잡한 명령어 집합을

In [14]:
ko_lima_multi_conv['text'][:5]

[[{'from': '사람',
   'value': '제가 읽은 대부분의 책은 <code>printf와</code> <code>scanf를</code> 사용하는 예제를 제공합니다. 어느 시점에서 학생들은 이 두 함수를 사용하는 방법을 완벽하게 알고 있지만, <code>stdin</code>, <code>stdout</code> 및 <code>argv에</code> 대해서는 알지 못합니다. \n저와 많은 프로그래밍 원칙(예: KISS)에 따르면 프로그램은 사용자에게 프롬프트를 표시하기 위해 실행을 중단해서는 안 됩니다. 대신, 이것은 훨씬 영리한 접근 방식이며 개발자는 파이프의 힘을 사용하고 옵션과 인수를 사용하는 방법을 배워야합니다. \n저는 이렇게 생각합니다:\n<code>$ 무슨날 1982 02 16\n월요일\n</code>\n보다 훨씬 낫습니다: \n<code> whatdaywas\n연도를 입력합니다: 1982\n월을 입력합니다: 2\n하루를 입력하세요: 16\n월요일이었습니다.\n</code>\n이러한 교육적 접근 방식에는 어떤 근거가 있나요?'},
  {'from': '어시스턴트',
   'value': '어린 CS 학생들을 가르칠 때 사용자 입력이라는 개념이 의외로 까다롭다는 걸 자주 느낍니다. 만약 사용자 입력이 전혀 없다면 프로그램의 결과가 미리 만들어져서 그냥 출력만 남아 있을 수도 있죠. 처음엔 argc/argv로 모든 입력을 받는 걸로 한 가지를 가르쳐줄 수 있지만, 그 다음부터는 컴퓨터가 그냥 돌아간다고만 생각하기 쉽습니다. 그런데 프로그램을 실행하는 도중에 입력을 받는 건 또 다른 교훈을 줍니다.\n\n사실 사용자 입력이라는 아이디어는 우리가 일상에서 쓰는 거의 모든 프로그램이 그렇게 동작한다는 점을 생각하면 직관적이어야 하는데, 의외로 그렇지 않습니다. 그래서 이걸 초기에 알려주면 컴퓨터가 어떻게 작동하는지에 대한 오해를 본격적인 문제로 되기 전에 정리하는 데 도움이 됩니다.\n\n물론 모든 학생이 똑같진 않지만, 공통적으로 나타나는 

In [15]:


def convert_to_conversational(example):
    # instruction과 input을 합쳐 user의 발화로 구성
    user_content = example["text"][0]
    model_response = example['text'][1]

    return {
        "prompt": [{"role": "user", "content": user_content}],
        "completion": [{"role": "assistant", "content": model_response}]
    }

# convert_to_conversational 적용

conver_dataset = ko_lima_conv.map(convert_to_conversational,
                                     remove_columns=ko_lima_conv.column_names
                                          )

In [16]:
conver_dataset['completion'][0]

[{'content': '이 질문은 꽤 넓은 범위를 다루고 있어서, 뇌가 뉴런만 있는 게 아니라 신경교세포(지지세포)와 유사분열 전의 신경 줄기세포까지 포함된다는 점을 같이 봐야 해요. 그리고 동료 과학자들이 지적했듯이, 발달 중인 배아의 뇌는 성인 뇌와 많이 다르기 때문에 발달 단계도 되게 중요합니다.\n\n하지만 여러 논문을 살펴보면 이 질문에 대한 답은 의외로 단순해요: 네, 뇌세포는 이동합니다. 성체 뇌에서는 아교세포가 이동하고(Klämbt, 2009), 그중에서도 희돌기아교세포같은 경우엔 절연 수초를 만들기 위해 비교적 먼 거리의 표적 축삭돌기를 찾아 이동하기도 합니다(Tsai and Miller, 2002). 신경 줄기세포도 손상에 반응해 장거리로 이동하고(Imitola et al., 2004), 해마나 뇌실하 영역 같은 특정 줄기세포 자리에서 다른 영역으로 이동하는 게 보고돼 있어요(Clarke, 2003). 유사분열 후지만 아직 분화되지 않은 뉴런들이 어류의 성체 뇌에서 이동하는 것(Scott et al., 2012)과 포유류·비인간 영장류에서도 이동하는 것이 관찰된 것도 같은 맥락입니다(Sawada et al., 2011).\n\n그리고 당연히 신경교세포, 줄기세포, 뉴런 모두 배아 발달 과정에서도 이동해요. 특히 말초 쪽으로 기능을 수행해야 하는 유사분열 후 뉴런들은 신경능선에서 목표 위치까지 비교적 먼 거리를 이동해야 합니다(신경과학, 2판, 뉴런의 이동).',
  'role': 'assistant'}]

In [17]:
ko_lima_multi_conv['text'][21]

[{'from': '사람',
  'value': '라틴어 수업에서 크리스토퍼라는 이름은 그리스어에 뿌리를 두고 있으며 "그리스도를 운반하는 사람"을 의미한다고 들었습니다. 여기서 라틴어는 운반하다라는 동사 페로(fero)에서 유래한 것으로 추정됩니다.\n그렇다면 제니퍼라는 이름도 비슷한 어원을 가지고 있을까요? 그렇다면 제니퍼는 무엇을 들고 있을까요?'},
 {'from': '어시스턴트',
  'value': '이건 언어적 동질화 때문에 생기는 잘못된 동족어의 전형적인 예야. 외국어를 들을 때 우리는 그 소리를 자기 말 소리에 맞춰 발음하려고 하고, 그래서 외국어의 소리들이 우리 귀에 자연스럽고 우리 목적에 맞는 소리로 대응돼 버려. 이런 식으로 서로 전혀 관련 없는 소리나 음절들이 한 가지 모국어 소리·음절로 합쳐져서, 결국 관련 없는 소리들을 관련 있는 소리처럼 듣게 되는 거지.\n\n예를 들어 중남미의 여러 언어에는 /v/ 소리가 없어서, 중앙아메리카 원주민들이 스페인어의 v를 들으면 익숙하지 않으니까 보통 b로 발음하곤 했어. b가 v와 비슷하게 들리니까 큰 장애는 아니었지만, 그 결과로 \'투표하다(votar)\'와 \'버리다(botar)\' 같은 단어들이 같은 소리로 들려서 말장난이 엄청 생겼어. 라틴아메리카 출신 스페인어 화자들이 영어를 배울 때도 "Thank you bery much"처럼 듣게 되는 경우가 흔한데, 여기서 \'very\'를 \'berry\'나 \'bury\'처럼 발음해도 이 단어들은 공통 조상이 있는 게 아니야.\n\n마지막으로 영어 철자 얘기해보자면, 영어권 사람들과 사전 편찬자들은 원래 철자를 가능한 한 보존하려는 경향이 있어서 철자 규칙이 통일되기 어렵고 예외가 많아. 예컨대 그리스어 접미사 -φορος는 보통 -pher로 옮겨오지만, Gwenhwyfar의 어미는 -fer로, aquifer의 어미도 로마식 표기를 따라 -fer로 남아 있지. 그래서 철자가 복잡하긴 해도 단어 자체에 어원 단서가 많이 숨어 있어서 라틴어·그리스

In [18]:
def convert_multi_turn(example):
    convs = example["text"]

    prompt_list = []
    completion = None

    # 각 턴을 순서대로 처리
    for i, turn in enumerate(convs):
        role = "user" if turn["from"] == "사람" else "assistant"
        entry = {"role": role, "content": turn["value"]}

        # 마지막이 assistant이면 그걸 completion으로, 나머지는 prompt에 저장
        if i == len(convs) - 1 and role == "assistant":
            completion = [entry]
        else:
            prompt_list.append(entry)

    # 만약 마지막이 human이면 모델 응답이 없는 incomplete sample이므로 그 전의 것을 completion으로 사용
    if completion is None:
        return {"prompt": prompt_list[:-2], "completion": [prompt_list[-2]]}

    return {"prompt": prompt_list, "completion": completion}

multiconver_dataset = ko_lima_multi_conv.map(convert_multi_turn,
                                     remove_columns=ko_lima_multi_conv.column_names
                                          )

In [19]:
multiconver_dataset

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 30
})

In [20]:
from datasets import concatenate_datasets

merged_dataset = concatenate_datasets([conver_dataset, multiconver_dataset])

print(merged_dataset)
print(len(merged_dataset))
print(merged_dataset[0])

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 1030
})
1030
{'prompt': [{'content': '뇌세포가 움직일 수 있나요? 여기서 이동이란 장거리 이동(가급적 뇌 내에서만)을 의미합니다.', 'role': 'user'}], 'completion': [{'content': '이 질문은 꽤 넓은 범위를 다루고 있어서, 뇌가 뉴런만 있는 게 아니라 신경교세포(지지세포)와 유사분열 전의 신경 줄기세포까지 포함된다는 점을 같이 봐야 해요. 그리고 동료 과학자들이 지적했듯이, 발달 중인 배아의 뇌는 성인 뇌와 많이 다르기 때문에 발달 단계도 되게 중요합니다.\n\n하지만 여러 논문을 살펴보면 이 질문에 대한 답은 의외로 단순해요: 네, 뇌세포는 이동합니다. 성체 뇌에서는 아교세포가 이동하고(Klämbt, 2009), 그중에서도 희돌기아교세포같은 경우엔 절연 수초를 만들기 위해 비교적 먼 거리의 표적 축삭돌기를 찾아 이동하기도 합니다(Tsai and Miller, 2002). 신경 줄기세포도 손상에 반응해 장거리로 이동하고(Imitola et al., 2004), 해마나 뇌실하 영역 같은 특정 줄기세포 자리에서 다른 영역으로 이동하는 게 보고돼 있어요(Clarke, 2003). 유사분열 후지만 아직 분화되지 않은 뉴런들이 어류의 성체 뇌에서 이동하는 것(Scott et al., 2012)과 포유류·비인간 영장류에서도 이동하는 것이 관찰된 것도 같은 맥락입니다(Sawada et al., 2011).\n\n그리고 당연히 신경교세포, 줄기세포, 뉴런 모두 배아 발달 과정에서도 이동해요. 특히 말초 쪽으로 기능을 수행해야 하는 유사분열 후 뉴런들은 신경능선에서 목표 위치까지 비교적 먼 거리를 이동해야 합니다(신경과학, 2판, 뉴런의 이동).', 'role': 'assistant'}]}


In [21]:
from trl import apply_chat_template

applied_template_dataset = merged_dataset.map(apply_chat_template,
                                     fn_kwargs={"tokenizer": tokenizer}
                                          )

In [22]:
applied_template_dataset[20]

{'prompt': '[|system|][|endofturn|]\n[|user|]전송 방법이 아닌 메시지를 암호화해야 한다면 왜 와이파이 보안에 신경을 써야 할까요? 이것은 단지 보안 연극일까요?\n[|assistant|]',
 'completion': '네트워크 암호화는 TLS 같은 애플리케이션 계층 암호화랑은 다른 위협으로부터 보호해줘.\n\n특히 와이파이 암호화 같은 네트워크 암호화는 주로 같은 로컬 네트워크에 있는 공격자를 막는 용도야. 예를 들어 누가 연결돼 있는지, 네트워크에 어떤 기기가 있는지 같은 패턴을 엿보는 걸 막고, ARP나 DNS 같은 낮은 수준의 메시지를 관찰하거나 변조하는 것도 방지하고, 네트워크에 있어서는 안 되는 기기에서 오는 브로드캐스트를 차단하고, 패킷 변조나 선택적 간섭으로부터도 보호하도록 설계된 거지.\n\n반면 TLS는 TCP/IP의 낮은 수준 패킷(예: 연결된 기기의 IP 주소 같은)을 보호해주지 않아. TLS 핸드셰이크 자체도 SNI(Server Name Indication)처럼 연결에 대한 여러 정보를 노출하도록 되어 있고.\n\n이런 설계상의 이유 때문에, 와이파이 하드웨어가 이미 암호화를 처리할 수 있는 코드랑 성능을 갖추고 있다면 필요한 패킷만 골라서 보호하려고 하기보다 모든 와이파이 패킷을 암호화해 버리는 게 더 쉬워. 또 덤으로, 네트워크 레벨에서 암호화하면 비보안 HTTP 연결이라도 적어도 인프라 제공업체가 아니라 같은 네트워크를 쓰는 다른 사용자들로부터는 보호되는 장점이 있어.\n\n정리하면, 네트워크 암호화는 네트워크 자체를 보호하려는 거고, 애플리케이션(서비스) 암호화는 서비스로의 연결을 보호하려는 거야. 둘은 서로 보완하지만, 어느 하나가 다른 하나를 완전히 대체하진 않아.[|endofturn|]\n'}

In [23]:
import re
import unicodedata

def fix_special_token_spacing(d):
    prompt = d["prompt"]
    completion = d["completion"]

#     # 유니코드 정규화
#     prompt = unicodedata.normalize("NFKC", prompt)
#     completion = unicodedata.normalize("NFKC", completion)

    # [|assistant|] 뒤에 문장부호가 붙은 경우 → 공백 추가
    prompt = prompt.replace("[|assistant|]", "[|assistant|]  ")

#     # invisible whitespace 정리
#     prompt = re.sub(r"[\u200b\u200c\u200d\u00a0]", " ", prompt)
#     completion = re.sub(r"[\u200b\u200c\u200d\u00a0]", " ", completion)

    return {"prompt": prompt, "completion": completion}

final_applied_template_dataset = applied_template_dataset.map(fix_special_token_spacing)

In [24]:
final_applied_template_dataset

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 1030
})

In [25]:
final_applied_template_dataset['prompt']

['[|system|][|endofturn|]\n[|user|]뇌세포가 움직일 수 있나요? 여기서 이동이란 장거리 이동(가급적 뇌 내에서만)을 의미합니다.\n[|assistant|]  ',
 '[|system|][|endofturn|]\n[|user|]컴퓨터 시스템 강의에서 MIPS 프로세서에 대해 소개했습니다. 이 프로세서는 한 학기 동안 (재)개발되었으며 실제로 이해하기 매우 쉬웠습니다. 기본 명령어가 규칙적으로 인코딩되어 있고 전선을 단순하게 유지하기 위해 명령어가 몇 개만 있는 RISC 설계를 사용합니다.\nCISC는 다른 철학을 따른다고 언급했습니다. x86 명령어 세트를 잠깐 살펴보고 충격을 받았습니다. 이렇게 복잡한 명령어 집합을 사용하는 프로세서를 어떻게 만들려고 하는지 상상할 수 없었습니다!\n그래서 저는 프로세서 시장의 상당 부분이 CISC 아키텍처를 사용하는 데에는 그럴 만한 이유가 있을 거라고 생각했습니다. 그게 뭔가요?\n[|assistant|]  ',
 '[|system|][|endofturn|]\n[|user|]명령줄에서 CSV와 같은 표 형식의 파일을 가로 및 세로 스크롤로 볼 수 있으면 좋을 것 같습니다.\n[|assistant|]  ',
 '[|system|][|endofturn|]\n[|user|]슬레이터형 궤도(STO)는 원자 및 분자 QM 계산에 가우시안형 궤도(GTO)보다 더 정확한 것으로 간주되는데, 그 이유는 $e^{-\\alpha r}$ 가 $r \\to \\infty$ 로 붕괴하기 때문입니다. 그러나 GTO는 계산하기 쉽기 때문에 더 많이 사용됩니다. GTO는 $e^{-\\alpha r^2}$ 로 붕괴하므로 가우시안 붕괴 동작을 보정하기 위해 GTO 기저 세트에 확산 함수를 추가하는 것이 적절할 때가 있습니다.\n또한 정확한 수소 파동 함수는 기하급수적으로 붕괴하므로 STO에 대한 동기가 있습니다.\n자유 공간에서 원자와 분자에 대한 슈뢰딩거 방정식을 풀기 위한 유일한 경계 요건은 파동 함수가 0이 되는 것( $r

In [26]:
for d in final_applied_template_dataset:
    tokenized_prompt = tokenizer(d['prompt'], add_special_tokens=False).input_ids
    tokenized_full = tokenizer(d['prompt'] + d['completion'], add_special_tokens=False).input_ids

    if not tokenized_full[:len(tokenized_prompt)] == tokenized_prompt:
        print(tokenizer.decode(tokenized_full[:len(tokenized_prompt)]))
        print(tokenizer.decode(tokenized_prompt))
        print(len(tokenized_prompt), len(tokenized_full[:len(tokenized_prompt)]), len(tokenized_full))
        print(d['prompt'] + d['completion'])
        print("========="*5)

In [27]:
tokenizer.decode(tokenizer(final_applied_template_dataset['prompt'][12] + final_applied_template_dataset['completion'][12]).input_ids)

'[|system|][|endofturn|]\n[|user|]중국은 일부러 통화를 평가절하하는데 터키는 통화가치 절하를 걱정하는 이유는 무엇인가요?\n[|assistant|]  무역 흑자/적자\n통화 가치를 평가절하한다는 건 보통 한 가지 효과를 뜻해요. 외국에서 물건을 사오는 건 더 비싸지고, 반대로 우리 제품은 세계시장에선 상대적으로 싸지게 되거든요. 그래서 물건을 많이 수출하는 나라는 통화 평가절하를 원하고, 수입을 많이 하는 나라는 굳이 그렇게 하려 하지 않는 거죠.\n\n예를 들어 2016년에 터키는 수입이 약 1,860억 달러에 수출이 약 1,560억 달러였어요. 그래서 약 19%의 무역적자를 기록했다는 얘기고요. 반면 중국은 같은 해 수입이 약 1조2,300억 달러, 수출이 약 2조2,700억 달러라서 약 84%의 무역흑자를 냈습니다. 이런 이유로 중국은 통화 평가절하에 더 매력을 느끼고, 터키는 그렇지 않은 거예요.\n\n부채 관리\n통화 가치를 떨어뜨리는 또 다른 이유는 그 통화로 표시된 부채의 실질 가치를 줄이기 위해서예요. 공공·민간 부채가 너무 많을 때, 심한 인플레이션을 통해 빚의 실질 부담을 낮추는 방법이 있거든요.\n\n숫자를 보면 차이가 확실해요. 중국의 GDP 대비 부채 비율은 약 47.6%인 반면 터키는 28.3% 수준이에요. 민간 부채 쪽을 보면 더 극단적이라, 터키의 민간부채는 GDP의 약 170%인데 중국은 300%가 넘는다고 알려져 있어요. 그래서 빚을 인플레이션으로 덜어내려는 유인은 터키보다 중국에서 더 강하게 보이는 겁니다.\n\n외국인 투자자 관계\n그럼 왜 모든 나라가 그냥 지폐에 0 몇 개 붙여서 빚을 없애버리지 않을까요? 통화를 함부로 부풀리면 외국인 투자자들이 떠나버릴 가능성이 크거든요. 몇 년 뒤 그 돈이 아무 가치가 없을지도 모른다면 누가 그 나라에 투자하려 하겠어요? 투자자들은 안정적인 통화를 원하니까요. 여기서도 터키와 중국의 동기가 달라요. 터키는 외국인 투자를 적극적으로 끌어들이는 쪽이고, 중국은 

### Training

In [28]:
from datasets import Dataset
from trl import SFTTrainer
from trl import SFTConfig


training_args = SFTConfig(
    output_dir="/home/work/Korean_Spoken_model/EXAONE3.5_7.8B_packed_SFT-INSTRUCT/",
    logging_dir= "/home/work/Korean_Spoken_model/EXAONE3.5_7.8B_packed_SFT-INSTRUCT_LOG/",
    deepspeed = "/home/work/Korean_Spoken_model/ds_config.json",  # DeepSpeed Zero 2 no offload optim
    num_train_epochs=10,
    learning_rate = 1e-5,
    bf16 =True,
    optim = "adamw_torch_fused",
    per_device_train_batch_size=2,
    gradient_checkpointing=False, 
 #   torch_compile=True,
    gradient_accumulation_steps = 4,
    logging_strategy = "epoch",
    save_strategy = "epoch",
    lr_scheduler_type = "linear",
    warmup_ratio=0.00,
    weight_decay=0.1,
  #  save_steps=10000,
 #   logging_steps=10000,
 #   save_total_limit =3,
    report_to="tensorboard",
 #   dataset_text_field = "Text",
    packing = False,
    chat_template_path = MODEL_NAME,
    completion_only_loss = True
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=final_applied_template_dataset,
    processing_class=tokenizer,
)

/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


In [29]:
print("Done")

Done


In [30]:
#trainer.add_callback(CustomCallback(trainer))
trainer.train()

/home/work/.local/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 800.00 MiB. GPU 0 has a total capacity of 79.19 GiB of which 657.50 MiB is free. Process 2166060 has 3.38 GiB memory in use. Including non-PyTorch memory, this process has 75.16 GiB memory in use. Of the allocated memory 72.10 GiB is allocated by PyTorch, and 1.58 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:


trainer.save_model("/home/work/Korean_Spoken_model/Final_LGEXAONE3.5_7.8B_packed_SFT-INSTRUCT")

